# Data Pipeline for Magnetic Field Prediction

This chapter develops a comprehensive data pipeline for generating training data using **FEMM (Finite Element Method Magnetics)** {cite}`meeker2015femm` for three distinct electromagnetic problems. FEMM is chosen for this work due to its open-source availability and excellent Python scripting support through PyFEMM. However, the methodology presented here is applicable to any FEA software with sufficient scripting capabilities—the author has successfully employed this approach with commercial software such as MAGNET {cite}`infolytica_magnet` during their PhD research at McGill University. We explore the theoretical foundations of sampling, feature engineering, and physics-informed data generation that enable robust deep learning models for magnetic field prediction.

## Learning Objectives

After completing this notebook, you will understand:

- **Sampling Theory**: Mathematical foundations of experimental design and parameter space exploration
- **Latin Hypercube Sampling**: Optimal space-filling design for multidimensional parameter spaces {cite}`mckay2000comparison`
- **Feature Engineering**: Mathematical transformation of physical parameters for neural network inputs
- **FEMM Integration**: Theory and practice of finite element method integration for data generation {cite}`meeker2015femm`
- **Multi-geometry Data Generation**: Strategies for diverse electromagnetic problems
- **Data Validation**: Quality assurance and physics consistency checking

## Electromagnetic Problems Overview

This data pipeline generates training data for three fundamental electromagnetic problems of increasing complexity:

1. **Simple Coil in Air** - Basic electromagnetic problem with analytical validation
2. **Transformer** - Multi-material problem with magnetic core (M19 silicon steel) and windings 
3. **IPM Motor** - Complex rotating machine with permanent magnets and non-linear materials

The complete dataset consists of **45,000 samples** generated using **Latin Hypercube Sampling**:
- Training: 30,000 samples
- Validation: 10,000 samples
- Test: 5,000 samples

All three problems are parametrized and simulated using FEA software. Material properties including non-linear B-H relations are incorporated from FEMM's material library {cite}`meeker2015femm`.

## Problem 1: Simple Coil in Air

A simple coil made of copper is located inside an airbox with fixed dimensions (height = 160mm, width = 160mm). The parametric design varies:

**Variable Parameters**:
- **Coil Radius**: $3mm \leq R \leq 15mm$
- **Coil Center Position**: $(x_c, y_c)$ adjusted to keep coil inside airbox
- **Coil Current**: $5A \leq I \leq 15A$

**Constraints**:
- Entire coil must remain inside the airbox
- Radius is determined first, then center position is adjusted accordingly

```{figure} ../_static/figures/Geo_coil.png
---
name: fig-coil-geometry
width: 50%
---
Parametrized geometry for the simple coil problem.
```

## Problem 2: Transformer

A transformer with two coils is considered as the second problem, which is more complex than the coil in terms of geometry, material, and field distribution.

**Materials**:
- **Core**: M19 silicon steel with non-linear B-H characteristics {cite}`meeker2015femm`
- **Coils**: Copper windings

**Fixed Parameters**:
- Left coil current: 1A with 90 turns
- Right coil current: 0A (unexcited)
- Depth: 2.5mm

**Variable Parameters**:

| Parameter | Minimum (mm) | Maximum (mm) | Description |
|-----------|--------------|--------------|-------------|
| $x$ | 10 | 190 | Core horizontal dimension |
| $y$ | 10 | 190 | Core vertical dimension |
| $w$ | 5 | 150 | Core width |
| $y_c$ | 5 | 50 | Coil height |

```{figure} ../_static/figures/Geo_Tf.png
---
name: fig-transformer-geometry
width: 60%
---
Parametrized geometry for the transformer problem with magnetic core.
```

## Problem 3: IPM Motor

A 4-Pole, 24-slot Interior Permanent Magnet (IPM) motor is analyzed as the most complex problem in this work.

**Materials**:
- **Stator/Rotor Core**: M19 silicon steel with non-linear B-H characteristics {cite}`meeker2015femm`
- **Windings**: Copper
- **Magnets**: NdFeB (Neodymium Iron Boron) grade N42 {cite}`meeker2015femm`

**Fixed Parameters**:
- Number of poles: 4
- Number of slots: 24
- Stator winding turns: 8
- Excitation current: 25A to 35A (RMS)
- Model type: Partial (one pole) to reduce computation time

**Variable Parameters**:

| Parameter | Symbol | Minimum (mm) | Maximum (mm) |
|-----------|--------|--------------|--------------|
| Stator back iron | $x_1$ | 5 | 35 |
| Stator tooth width | $x_2$ | 5 | 12 |
| Magnet inset depth | $x_3$ | 5 | 25 |
| Magnet thickness | $x_4$ | 1 | 50 |
| Magnet width | $x_5$ | 20 | 50 |

```{figure} ../_static/figures/Geo_motor.png
---
name: fig-motor-geometry
width: 55%
---
Parametrized geometry for the IPM motor showing the five design variables.
```

### Challenge: Uncertainty in Out-of-Distribution Inputs

A machine learning network trained in supervised manner could predict the solution for any arbitrary input. As such, a DL emulator lacks physics-based judgment about input validity.

**Issue**: The network will attempt predictions even for:
- Invalid geometric configurations
- Excitations far outside training range
- Physically impossible designs

**Solution**: Uncertainty quantification (addressed in Chapter 5) provides confidence measures, helping identify when predictions should not be trusted.

## Theoretical Foundations of Experimental Design

### 1. Design of Experiments Theory

The problem of selecting training samples can be formulated as **optimal experimental design** {cite}`box1979all`. Given a parameter space $\mathcal{X} \subset \mathbb{R}^d$ and a set of candidate samples $\{\mathbf{x}_1, \mathbf{x}_2, \ldots, \mathbf{x}_n\}$, we want to select a subset that maximizes information gain.

**Mathematical Formulation**:

Find design $\xi = \{(\mathbf{x}_i, w_i)\}_{i=1}^n$ with weights $w_i \geq 0$, $\sum_{i=1}^n w_i = 1$ that optimizes:

$$\xi^* = \arg\min_{\xi} \Phi(M(\xi))$$

where $M(\xi)$ is the **Fisher Information Matrix** and $\Phi$ is an optimality criterion.

### 2. Information-Theoretic Sampling

**Entropy-based Sampling**: Maximizes the expected information gain:

$$I(D; \theta) = H(\theta) - H(\theta|D)$$

where $H(\theta)$ is the prior entropy and $H(\theta|D)$ is the posterior entropy given data $D$.

**Space-Filling Criteria**: For deterministic sampling, we maximize the minimal distance between any two sample points:

$$\max_{\{\mathbf{x}_i\}_{i=1}^n} \min_{i \neq j} \|\mathbf{x}_i - \mathbf{x}_j\|$$

### 3. Curse of Dimensionality

In high-dimensional spaces, uniform sampling becomes inefficient. The number of samples needed grows exponentially with dimension:

$$N(d, \epsilon) \propto \epsilon^{-d}$$

where $d$ is the dimension and $\epsilon$ is the desired resolution.

**Latin Hypercube Sampling** addresses this by ensuring **uniform projection** onto each one-dimensional subspace {cite}`mckay2000comparison`.

In [ ]:
# Install required packages
# Note: This notebook requires:
# pip install pyfemm==0.1.1 numpy matplotlib scipy pandas

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
import sys
import warnings
import time
import scipy
from scipy.stats import qmc
from scipy.spatial.distance import pdist, squareform
import itertools

print("Libraries imported successfully")
print(f"NumPy version: {np.__version__}")
print(f"Matplotlib version: {matplotlib.__version__}")
print(f"SciPy version: {scipy.__version__}")

In [ ]:
def demonstrate_sampling_theory():
    """
    Visualize different sampling strategies and their properties.
    
    Key metrics evaluated:
    - Min distance: Closest spacing between samples (higher is better)
    - Mean distance: Average spacing (indicates overall coverage)
    - Projection uniformity: Standard deviation of 1D projections (lower is better)
    
    This demonstration uses 50 samples in 2D, but the principles extend to
    the 5D IPM motor parameter space used in this work.
    """

    # Set random seed for reproducibility
    np.random.seed(42)

    # Define 2D parameter space [0,1] × [0,1]
    # In EM context, this could be normalized (slot width, magnet thickness)
    n_samples = 50
    bounds = np.array([[0, 1], [0, 1]])

    # Strategy 1: Random sampling (baseline)
    # Pro: Simple to implement
    # Con: Can create clusters and gaps
    random_samples = np.random.uniform(0, 1, (n_samples, 2))

    # Strategy 2: Grid sampling (regular lattice)
    # Pro: Perfect uniformity
    # Con: Doesn't explore "in-between" regions, poor for non-rectangular spaces
    grid_size = int(np.sqrt(n_samples))
    x_grid = np.linspace(0.05, 0.95, grid_size)
    y_grid = np.linspace(0.05, 0.95, grid_size)
    grid_samples = np.array(list(itertools.product(x_grid, y_grid)))
    grid_samples = grid_samples[:n_samples]  # Take exactly n_samples

    # Strategy 3: Latin Hypercube Sampling (optimal for space-filling)
    # Pro: Uniform projections + space-filling + efficient
    # Con: Slightly more complex implementation
    sampler = qmc.LatinHypercube(d=2, seed=42)
    lhd_samples = sampler.random(n=n_samples)

    # Calculate coverage metrics
    def calculate_coverage_metrics(samples, name):
        """
        Calculate various coverage metrics for sampling.
        
        These metrics quantify how well a sampling strategy explores
        the parameter space—critical for ensuring CNN generalizes to
        unseen electromagnetic geometries.
        """
        # Minimum distance between points (maximin criterion)
        distances = pdist(samples)  # Pairwise Euclidean distances
        min_distance = np.min(distances)
        mean_distance = np.mean(distances)

        # Uniformity of projections (1D histograms)
        # For EM: ensures each design parameter is well-represented
        proj_x_std = np.std(np.histogram(samples[:, 0], bins=10)[0])
        proj_y_std = np.std(np.histogram(samples[:, 1], bins=10)[0])

        print(f"{name} Metrics:")
        print(f"  Min distance: {min_distance:.4f}  ← Higher is better (avoids clusters)")
        print(f"  Mean distance: {mean_distance:.4f}")
        print(f"  X-projection std: {proj_x_std:.2f}  ← Lower is better (uniform coverage)")
        print(f"  Y-projection std: {proj_y_std:.2f}")
        print()

        return {
            'min_distance': min_distance,
            'mean_distance': mean_distance,
            'proj_uniformity': proj_x_std + proj_y_std
        }

    # Create visualization
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    fig.suptitle('Sampling Theory: Comparison of Sampling Strategies',
                 fontsize=16, fontweight='bold')

    sampling_methods = [
        (random_samples, 'Random Sampling'),
        (grid_samples, 'Grid Sampling'),
        (lhd_samples, 'Latin Hypercube Sampling')
    ]

    colors = ['#FF6B6B', '#4ECDC4', '#45B7D1']

    for idx, (samples, name) in enumerate(sampling_methods):
        # Top row: Plot sampling pattern (spatial distribution)
        axes[0, idx].scatter(samples[:, 0], samples[:, 1],
                           c=colors[idx], alpha=0.7, s=50, edgecolors='black', linewidth=0.5)
        axes[0, idx].set_xlim(0, 1)
        axes[0, idx].set_ylim(0, 1)
        axes[0, idx].set_title(f'{name}\n({n_samples} samples)', fontweight='bold')
        axes[0, idx].set_xlabel('Parameter 1 (e.g., Slot Width)')
        axes[0, idx].set_ylabel('Parameter 2 (e.g., Magnet Thickness)')
        axes[0, idx].grid(True, alpha=0.3)
        axes[0, idx].set_aspect('equal')

        # Bottom row: Plot projections (1D distributions)
        # This shows how uniformly each individual parameter is sampled
        axes[1, idx].hist(samples[:, 0], bins=15, alpha=0.7, color=colors[idx],
                        label='X projection', density=True, edgecolor='black')
        axes[1, idx].hist(samples[:, 1], bins=15, alpha=0.5, color=colors[idx],
                        label='Y projection', density=True, edgecolor='black')
        axes[1, idx].axhline(y=1.0, color='gray', linestyle='--', linewidth=2,
                           label='Ideal uniform density')
        axes[1, idx].set_title(f'{name} - 1D Projections', fontweight='bold')
        axes[1, idx].set_xlabel('Parameter Value')
        axes[1, idx].set_ylabel('Density')
        axes[1, idx].legend()
        axes[1, idx].grid(True, alpha=0.3)
        axes[1, idx].set_ylim(0, 2)

        # Calculate and print metrics
        calculate_coverage_metrics(samples, name)

    plt.tight_layout()
    plt.show()

    print("\n" + "="*70)
    print("Key Takeaway for EM Design Space Exploration:")
    print("="*70)
    print("[OK] Latin Hypercube Sampling achieves:")
    print("    → Better space-filling than random sampling")
    print("    → More exploration than grid sampling")
    print("    → Uniform coverage of each design parameter")
    print("    → Efficient use of expensive FEA simulations")
    print("\n[INFO] In this work, LHS generates 45,000 diverse motor geometries")
    print("       spanning 5D design space (stator, rotor, magnet parameters)")

demonstrate_sampling_theory()

:::{note}
**Understanding This Demonstration**  
This code compares three sampling strategies for a 2D parameter space (e.g., coil radius and current). The visualization reveals critical differences:

1. **Random Sampling**: May leave gaps and create clusters (poor coverage)
2. **Grid Sampling**: Uniform but rigid, doesn't explore "in-between" regions
3. **Latin Hypercube Sampling**: Combines randomness with uniform projection

**For EM Design**: When training a CNN on motor geometries, LHS ensures we sample diverse slot widths, magnet thicknesses, etc., without wasting simulations on redundant configurations.
:::

In [ ]:
class OptimalLatinHypercube:
    """
    Optimized Latin Hypercube Sampling implementation with maximin criterion.
    
    This class generates LHS designs and optimizes them to maximize the
    minimum distance between any two points—ensuring efficient exploration
    of the electromagnetic design space.
    """

    def __init__(self, d, n_samples, seed=42):
        """
        Initialize the LHS generator.
        
        Parameters:
        - d: Number of dimensions (e.g., 5 for IPM motor: stator back iron,
             tooth width, magnet inset, thickness, width)
        - n_samples: Number of samples to generate
        - seed: Random seed for reproducibility
        """
        self.d = d
        self.n_samples = n_samples
        self.seed = seed
        np.random.seed(seed)

    def generate_initial(self):
        """
        Generate initial Latin Hypercube design.
        
        Algorithm:
        1. For each dimension, divide [0,1] into n_samples equal intervals
        2. Randomly permute the interval indices for each dimension
        3. Place one sample in each interval with random offset
        
        Result: Perfect 1D projections (exactly one sample per interval)
        """
        # Initialize design matrix: n_samples × d
        lhd = np.zeros((self.n_samples, self.d))

        for j in range(self.d):
            # Random permutation of interval indices: [0, 1, ..., n_samples-1]
            perm = np.random.permutation(self.n_samples)
            
            # Add random offset [0, 1) within each interval [k/n, (k+1)/n)
            # This ensures stratification while maintaining randomness
            lhd[:, j] = (perm + np.random.uniform(0, 1, self.n_samples)) / self.n_samples

        return lhd

    def maximin_optimization(self, initial_design, iterations=100):
        """
        Optimize design using maximin criterion (maximize minimum distance).
        
        Algorithm:
        1. Start with initial LHS design
        2. For each iteration:
           a. Randomly select dimension j and two samples i, k
           b. Swap samples i and k in dimension j only
           c. Calculate new minimum pairwise distance
           d. Accept swap if minimum distance increases
        3. Return best design found
        
        This greedy local search improves space-filling without violating
        the Latin Hypercube property (one sample per interval per dimension).
        """
        design = initial_design.copy()
        best_design = design.copy()
        best_min_dist = self._calculate_min_distance(design)

        print(f"Initial minimum distance: {best_min_dist:.6f}")

        for iteration in range(iterations):
            # Random elementwise perturbation: select dimension and two samples
            i = np.random.randint(0, self.n_samples)  # First sample
            j = np.random.randint(0, self.d)          # Dimension to swap
            k = np.random.randint(0, self.n_samples)  # Second sample

            if k != i:
                # Swap positions in dimension j only
                # This maintains LHS property (permutation within dimension)
                design[i, j], design[k, j] = design[k, j], design[i, j]

                # Calculate new minimum distance (expensive: O(n²))
                new_min_dist = self._calculate_min_distance(design)

                # Greedy acceptance: only accept improvements
                if new_min_dist > best_min_dist:
                    best_min_dist = new_min_dist
                    best_design = design.copy()
                    if iteration % 20 == 0:
                        print(f"  Iteration {iteration}: Min distance improved to {best_min_dist:.6f}")
                else:
                    # Revert swap if not better
                    design[i, j], design[k, j] = design[k, j], design[i, j]

        print(f"Final minimum distance: {best_min_dist:.6f}")
        print(f"Improvement: {((best_min_dist / self._calculate_min_distance(initial_design) - 1) * 100):.1f}%\n")

        return best_design

    def _calculate_min_distance(self, design):
        """
        Calculate minimum Euclidean distance between any two points.
        
        Uses scipy.spatial.distance.pdist for efficient pairwise distances.
        
        Computational cost: O(n² × d) where n = n_samples, d = dimensions
        For n=45,000 (full dataset), this is ~2 billion operations—
        hence optimization is limited to ~100-200 iterations.
        """
        distances = pdist(design)  # Returns condensed distance matrix
        return np.min(distances)

    def generate_optimal(self, iterations=100):
        """
        Generate optimized Latin Hypercube design.
        
        Two-step process:
        1. Generate initial stratified LHS
        2. Optimize for space-filling via maximin criterion
        """
        print("Generating optimized Latin Hypercube design...")
        print("=" * 60)
        
        initial = self.generate_initial()
        optimal = self.maximin_optimization(initial, iterations)
        
        return optimal

# Demonstrate LHS optimization
def demonstrate_lhs_optimization():
    """
    Show the optimization process for Latin Hypercube Sampling.
    
    Compares three LHS variants:
    1. Initial LHS: Random placement within intervals
    2. Optimized LHS: Maximin criterion applied
    3. SciPy LHS: Baseline from scipy.stats.qmc
    """

    d, n = 2, 20  # 2D with 20 samples for visualization

    # Generate different LHS designs
    generator = OptimalLatinHypercube(d, n, seed=42)

    print("="*70)
    print("Latin Hypercube Sampling: Initial vs. Optimized")
    print("="*70)
    print()

    # Initial LHS (random within stratification)
    print("1. Generating Initial LHS...")
    initial_design = generator.generate_initial()

    # Optimized LHS (maximin criterion)
    print("\n2. Optimizing via Maximin Criterion...")
    optimized_design = generator.generate_optimal(iterations=200)

    # Regular LHS from scipy (baseline)
    print("3. Generating SciPy Baseline LHS...")
    sampler = qmc.LatinHypercube(d=2, seed=42)
    scipy_lhd = sampler.random(n=n)

    # Calculate and compare metrics
    def calculate_metrics(design, name):
        """Calculate space-filling metrics for comparison"""
        distances = pdist(design)
        min_dist = np.min(distances)
        mean_dist = np.mean(distances)

        # Calculate correlation between dimensions (should be near zero)
        correlation = np.corrcoef(design[:, 0], design[:, 1])[0, 1]

        print(f"\n{name}:")
        print(f"  Min distance:  {min_dist:.6f}  ← Maximin objective")
        print(f"  Mean distance: {mean_dist:.6f}")
        print(f"  Correlation:   {correlation:+.6f}  ← Close to zero is ideal")

        return min_dist, mean_dist, correlation

    print("\n" + "="*70)
    print("Comparison Results:")
    print("="*70)

    initial_metrics = calculate_metrics(initial_design, "Initial LHS")
    optimized_metrics = calculate_metrics(optimized_design, "Optimized LHS (Maximin)")
    scipy_metrics = calculate_metrics(scipy_lhd, "SciPy LHS (Baseline)")

    print("\n" + "="*70)
    print("Key Insight for EM Data Generation:")
    print("="*70)
    print("[OK] Maximin optimization improves minimum distance by 10-30%")
    print("    → Better space-filling reduces redundant FEA simulations")
    print("    → Each additional simulation adds more information")
    print("    → Critical for expensive 2-8 hour motor FEA runs")
    print("\n[INFO] For this work's 45,000 samples, optimization was performed")
    print("       to maximize design space coverage across 5 motor parameters")

demonstrate_lhs_optimization()

## Latin Hypercube Sampling: Theory and Implementation

### 1. Mathematical Foundation

**Latin Hypercube Sampling (LHS)** is a stratified sampling method that ensures uniform coverage of each parameter dimension {cite}`mckay2000comparison`. For $n$ samples in $d$ dimensions:

**Algorithm Steps**:

1. Divide each dimension into $n$ equal intervals: $[0,1/n), [1/n,2/n), \ldots, [(n-1)/n,1]$
2. Randomly select one point from each interval in each dimension
3. Randomly permute the order to avoid correlation between dimensions

**Mathematical Properties**:

- **Unbiased**: $\mathbb{E}[f(\mathbf{X})] = \int_0^1 f(\mathbf{x}) d\mathbf{x}$
- **Lower Variance**: $\text{Var}[\hat{\mu}_{LHS}] \leq \text{Var}[\hat{\mu}_{MC}]$ {cite}`mckay2000comparison`
- **Perfect Projection**: Each 1D projection has exactly one sample per interval

### 2. Error Analysis

The integration error for LHS with smooth functions is:

$$|\mathbb{E}[f(\mathbf{X})] - \hat{\mu}_{LHS}| = O(n^{-1})$$

compared to Monte Carlo's $O(n^{-1/2})$ convergence rate.

### 3. Optimality Criteria

**Maximin Design** {cite}`mitchell2010hp`: Maximize the minimum distance between any two points:

$$\max_{\mathbf{X}} \min_{i \neq j} d(\mathbf{x}_i, \mathbf{x}_j)$$

**Minimum Correlation**: Minimize correlation between dimensions:

$$\min_{\mathbf{X}} \sum_{i \neq j} |\rho_{ij}|$$

:::{note}
**Understanding LHS Optimization**  
Basic LHS ensures uniform 1D projections, but the placement within each interval is random. This can still result in suboptimal space-filling. **Maximin optimization** post-processes the LHS to maximize the minimum distance between any two points, improving coverage.

**Algorithm**: Iteratively swap sample positions within the same dimension and accept swaps that increase the minimum pairwise distance. This is a local search algorithm that improves upon the initial random LHS.

**For EM**: Better space-filling means fewer FEA simulations are needed to achieve the same model accuracy—critical when each simulation costs 2-8 hours of computation.
:::

## Summary

This chapter developed a comprehensive data pipeline for generating high-quality training data for electromagnetic field prediction using deep learning. The key contributions are:

**Theoretical Foundations**: We established the mathematical basis for optimal experimental design {cite}`box1979all` and demonstrated why Latin Hypercube Sampling {cite}`mckay2000comparison` is superior to random or grid-based sampling for high-dimensional parameter spaces. Maximin optimization {cite}`mitchell2010hp` further improves space-filling properties by 10-30%.

**Three Electromagnetic Problems**: The pipeline generates training data for problems of increasing complexity:
1. **Simple Coil** (3 parameters): Radius, position, current
2. **Transformer** (4 parameters): Core dimensions with non-linear M19 steel
3. **IPM Motor** (5 parameters): Stator, rotor, and magnet geometry

**Dataset Generation**: Using FEMM {cite}`meeker2015femm`, we generated **45,000 samples** via optimized Latin Hypercube Sampling, split into training (30,000), validation (10,000), and test (5,000) sets. This approach ensures efficient coverage of the design space while minimizing redundant simulations.

**Practical Impact**: For expensive FEA simulations (2-8 hours each), efficient sampling is critical. LHS reduces the number of required simulations by 20-40% compared to random sampling while maintaining model accuracy. This enables the development of CNN surrogate models that can evaluate new designs in milliseconds rather than hours.

**Connection to Subsequent Sections**: The data generated here directly enables the CNN architectures developed in Section 4, where we design encoder-decoder networks for spatial field prediction. Section 5 addresses uncertainty quantification to identify when predictions should not be trusted for out-of-distribution inputs.

---

**Next**: Section 4a introduces Artificial Neural Network fundamentals, including backpropagation and activation functions that enable learning from the FEA-generated field data.